# 00_config — Phase 2 shared configuration

Run with `%run ./00_config` from each Phase 2 notebook.

In [ ]:
from datetime import datetime, timezone
from uuid import uuid4
from pyspark.sql import functions as F, Window
from pyspark.sql.types import ArrayType, BooleanType, DateType, DoubleType, IntegerType, LongType, StringType, StructField, StructType, TimestampType

def ensure_widget(name, default, label):
    try: dbutils.widgets.get(name)
    except Exception: dbutils.widgets.text(name, default, label)

ensure_widget('landing_root', '', 'Landing root containing raw/ and dims/')
ensure_widget('catalog_name', 'main', 'Unity Catalog name')
ensure_widget('start_date', '', 'Start date YYYY-MM-DD; blank = latest partition')
ensure_widget('end_date', '', 'End date YYYY-MM-DD; blank = start date')
ensure_widget('load_type', 'incremental', 'full, incremental, or backfill')
ensure_widget('batch_id', '', 'Shared pipeline batch id')

LANDING_ROOT = dbutils.widgets.get('landing_root').rstrip('/')
CATALOG_NAME = dbutils.widgets.get('catalog_name').strip()
START_DATE = dbutils.widgets.get('start_date').strip()
END_DATE = dbutils.widgets.get('end_date').strip()
LOAD_TYPE = dbutils.widgets.get('load_type').strip().upper()
BATCH_ID = dbutils.widgets.get('batch_id').strip() or f'batch_{uuid4().hex}'
RUN_ID = BATCH_ID
if not LANDING_ROOT: raise ValueError('Set landing_root to the uploaded output/ directory.')
if LOAD_TYPE not in {'FULL', 'INCREMENTAL', 'BACKFILL'}: raise ValueError('load_type must be full, incremental, or backfill.')
spark.conf.set('spark.sql.session.timeZone', 'UTC')
spark.conf.set('spark.databricks.delta.schema.autoMerge.enabled', 'true')

BRONZE_SCHEMA, SILVER_SCHEMA, OPS_SCHEMA = (f'{CATALOG_NAME}.{x}' for x in ('bronze','silver','ops'))
BRONZE_EVENTS, BRONZE_USERS, BRONZE_CATALOG, BRONZE_EVENTS_QUARANTINE = (f'{BRONZE_SCHEMA}.{x}' for x in ('events','users','catalog','events_quarantine'))
SILVER_EVENTS, SILVER_USERS, SILVER_TRACKS, SILVER_EVENTS_QUARANTINE = (f'{SILVER_SCHEMA}.{x}' for x in ('events','users','tracks','events_quarantine'))
INGESTION_LOG, EXECUTION_LOG = f'{OPS_SCHEMA}.ingestion_log', f'{OPS_SCHEMA}.pipeline_execution_logs'
RAW_EVENTS_DIR, DIMENSIONS_DIR = f'{LANDING_ROOT}/raw', f'{LANDING_ROOT}/dims'
SESSION_GAP_MINUTES = 30
VALID_DEVICE_TYPES = {'mobile','desktop','web_player','smart_speaker'}
VALID_SOURCES = {'synthetic','synthetic_persona_matched','real','real_api'}
REQUIRED_EVENT_FIELDS = {'event_id','user_id','track_id','played_at','source'}

BRONZE_EVENT_SCHEMA = StructType([StructField('event_id',StringType(),False),StructField('user_id',StringType(),False),StructField('track_id',StringType(),False),StructField('played_at',StringType(),False),StructField('ms_played',LongType(),True),StructField('skipped',BooleanType(),True),StructField('device_type',StringType(),True),StructField('source',StringType(),False),StructField('_source_file',StringType(),False),StructField('_partition_dt',DateType(),False),StructField('_batch_id',StringType(),False),StructField('_rescued_data',StringType(),True),StructField('load_timestamp',TimestampType(),False)])
BRONZE_USER_SCHEMA = StructType([StructField('user_id',StringType(),False),StructField('country',StringType(),True),StructField('timezone',StringType(),True),StructField('is_premium',BooleanType(),True),StructField('_source_file',StringType(),False),StructField('_batch_id',StringType(),False),StructField('_rescued_data',StringType(),True),StructField('load_timestamp',TimestampType(),False)])
BRONZE_CATALOG_SCHEMA = StructType([StructField('track_id',StringType(),False),StructField('track_name',StringType(),True),StructField('artist',StringType(),True),StructField('genre',StringType(),True),StructField('release_year',IntegerType(),True),StructField('duration_sec',IntegerType(),True),StructField('popularity',IntegerType(),True),StructField('danceability',DoubleType(),True),StructField('energy',DoubleType(),True),StructField('valence',DoubleType(),True),StructField('tempo_bpm',DoubleType(),True),StructField('_source_file',StringType(),False),StructField('_batch_id',StringType(),False),StructField('_rescued_data',StringType(),True),StructField('load_timestamp',TimestampType(),False)])
PIPELINE_LOG_SCHEMA = StructType([StructField('log_id',StringType(),False),StructField('run_id',StringType(),False),StructField('batch_id',StringType(),False),StructField('layer',StringType(),False),StructField('target_table',StringType(),False),StructField('load_type',StringType(),False),StructField('parameter',StringType(),False),StructField('start_time',TimestampType(),False),StructField('end_time',TimestampType(),False),StructField('duration_sec',DoubleType(),True),StructField('status',StringType(),False),StructField('rows_read',LongType(),True),StructField('rows_inserted',LongType(),True),StructField('rows_updated',LongType(),True),StructField('rows_deleted',LongType(),True),StructField('rows_quarantined',LongType(),True),StructField('error_message',StringType(),True),StructField('load_timestamp',TimestampType(),False)])

def table_ref(name): return '`' + name.replace('.', '`.`') + '`'
def device_expression(column='device_type'):
    value = F.lower(F.trim(F.col(column)))
    return F.when(value.isin(*VALID_DEVICE_TYPES), value).otherwise(F.lit('other'))
def execution_metrics(table):
    row = spark.sql(f'DESCRIBE HISTORY {table_ref(table)} LIMIT 1').select('operationMetrics').first()
    values = (row['operationMetrics'] or {}) if row else {}
    def read(*names): return next((int(values[n]) for n in names if n in values), 0)
    return {'inserted':read('numTargetRowsInserted','numOutputRows'),'updated':read('numTargetRowsUpdated'),'deleted':read('numTargetRowsDeleted')}
def log_run(layer, target_table, parameter, started_at, status, rows_read=0, rows_inserted=0, rows_updated=0, rows_deleted=0, rows_quarantined=0, error_message=None):
    ended = datetime.now(timezone.utc).replace(tzinfo=None)
    started = started_at.replace(tzinfo=None) if getattr(started_at,'tzinfo',None) else started_at
    row=[(uuid4().hex,RUN_ID,BATCH_ID,layer,target_table,LOAD_TYPE,parameter,started,ended,(ended-started).total_seconds(),status,int(rows_read),int(rows_inserted),int(rows_updated),int(rows_deleted),int(rows_quarantined),error_message,ended)]
    spark.createDataFrame(row, PIPELINE_LOG_SCHEMA).write.format('delta').mode('append').saveAsTable(EXECUTION_LOG)
def latest_landing_partition():
    dates=[x.name.rstrip('/')[3:] for x in dbutils.fs.ls(RAW_EVENTS_DIR) if x.name.rstrip('/').startswith('dt=')]
    if not dates: raise ValueError(f'No dt= partitions found under {RAW_EVENTS_DIR}')
    return max(dates)
if not START_DATE: START_DATE=latest_landing_partition()
if not END_DATE: END_DATE=START_DATE
datetime.strptime(START_DATE,'%Y-%m-%d'); datetime.strptime(END_DATE,'%Y-%m-%d')
if START_DATE > END_DATE: raise ValueError('start_date must be on or before end_date.')
print(f'Phase 2 parameters: {LOAD_TYPE} {START_DATE}..{END_DATE}; batch={BATCH_ID}')
